# 🛡️ Lexsi Agent Circuit Defense: Colab Execution Notebook
### Provable Indirect Prompt Injection Defense via Subspace Weight Surgery
**Hackathon:** Lexsi AI Safety & Alignment Hackathon (Track 2: AI Red Teaming, Safety & Alignment)
**Toolchain:** `CuratorKIT` → `CircuitKIT` → `SafeTune` → `AuditKIT`


### 1. Verify GPU Hardware


In [ ]:
!nvidia-smi


### 2. Pull Latest Code & Install Dependencies


In [ ]:
import os
if os.path.basename(os.getcwd()) != "lexsi-agent-circuit-defense":
    if not os.path.exists("lexsi-agent-circuit-defense"):
        !git clone https://github.com/GiGiKoneti/lexsi-agent-circuit-defense.git
    %cd lexsi-agent-circuit-defense
!git pull origin main

!pip install -q transformers>=4.45.0 accelerate peft safetensors sentencepiece datasets plotly gradio pyyaml tqdm click
!pip install -q git+https://github.com/Lexsi-Labs/CuratorKIT.git || true
!pip install -q git+https://github.com/Lexsi-Labs/CircuitKIT.git || true
!pip install -q git+https://github.com/Lexsi-Labs/SafeTune.git || true
!pip install -q git+https://github.com/Lexsi-Labs/AuditKIT.git || true


### 3. Hugging Face Authentication
> ⚠️ **IMPORTANT: First-Time Setup Requirement**
> Both models are gated by Cohere on Hugging Face (`gated: auto`). You MUST accept conditions on both pages before running:
> 1. Open and click **Accept / Acknowledge conditions**: https://huggingface.co/CohereLabs/tiny-aya-global
> 2. Open and click **Accept / Acknowledge conditions**: https://huggingface.co/CohereLabs/aya-expanse-8b
> 3. Paste your Hugging Face Access Token below:


In [ ]:
from huggingface_hub import login
from getpass import getpass
hf_token = getpass("Enter your Hugging Face Token: ")
login(token=hf_token)


### 4. Choose Model Configuration


In [ ]:
# Choose model:
# For T4 fast run: "configs/model_tiny_aya.yaml"
# For full 8B model: "configs/model_aya_8b.yaml"
CONFIG_PATH = "configs/model_tiny_aya.yaml"
BASE_MODEL = "CohereLabs/tiny-aya-global" if "tiny" in CONFIG_PATH else "CohereLabs/aya-expanse-8b"
print(f"Target Config: {CONFIG_PATH}")
print(f"Target Model:  {BASE_MODEL}")


### 5. Step 1: CuratorKIT Triplet Curation & Preprocessing


In [ ]:
!python scripts/01_generate_triplets.py
!python scripts/preprocess_and_validate.py


### 6. Step 2: CircuitKIT EAP-IG Circuit Discovery & Subspace Extraction


In [ ]:
!python scripts/02_run_circuit_discovery.py --config {CONFIG_PATH} --triplets_path data/processed/triplets_cleaned.jsonl --output_basis data/processed/basis_matrices_k4.pt --num_samples 50


### 7. Step 3: SafeTune Orthogonal Weight Surgery ($P^\perp$)


In [ ]:
!python scripts/03_apply_weight_surgery.py --config {CONFIG_PATH} --basis_path data/processed/basis_matrices_k4.pt --output_dir safetune_model_projected


### 8. Step 4: AuditKIT Comprehensive Evaluation


In [ ]:
!python scripts/04_run_full_audit.py --base_model {BASE_MODEL} --edited_model ./safetune_model_projected --triplets_path data/processed/triplets_cleaned.jsonl --output_report data/processed/audit_report.json --num_eval_samples 50


### 9. View AuditKIT Benchmark Results


In [ ]:
import json
import pandas as pd
from IPython.display import display

with open("data/processed/audit_report.json") as f:
    report = json.load(f)

key = "Evaluation_Summary" if "Evaluation_Summary" in report else "Pareto_Frontier_Summary"
df = pd.DataFrame(report[key])
display(df)
